# Alignment pipeline (debug, fixed)

Robust discovery of episodes_audio by scanning upward through parent directories. Prints cwd, chosen episodes dir and found .webm files.

In [1]:
!git clone https://github.com/Yi-Star32/Video_Analyzer.git

Cloning into 'Video_Analyzer'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 31 (delta 0), reused 31 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 29.92 KiB | 3.32 MiB/s, done.


In [2]:
%cd Video_Analyzer

/content/Video_Analyzer


In [3]:
!pip install -r requirements.txt

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.13; 0.2.0 Requires-Python >=3.13; 0.2.1 Requires-Python >=3.13; 0.2.2 Requires-Python >=3.13
ERROR: Could not find a version that satisfies the requirement audioop-lts==0.2.2 (from versions: none)
ERROR: No matching distribution found for audioop-lts==0.2.2


In [4]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 21.5 MB/s eta 0:00:00


In [ ]:
!pip install git+https://github.com/m-bain/whisperx.git

  Cloning https://github.com/m-bain/whisperx.git to /tmp/pip-req-build-9t5dqf8r
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git /tmp/pip-req-build-9t5dqf8r
  Resolved https://github.com/m-bain/whisperx.git to commit 3ccc17b8de34f305300f8a3fd3c9f76ba820c0d0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with str

In [ ]:
from audio_matcher import alignment, chunking, embedding, index, io, phonemes

In [ ]:
from pathlib import Path
import warnings
from tqdm import TqdmWarning

# suppress tqdm widget warning in environments without ipywidgets
warnings.filterwarnings("ignore", category=TqdmWarning)

from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio

# target song (adjust if needed)
SONG_PATH = Path("../separated/htdemucs/audio/vocals.wav")


In [ ]:
# Robust upward search for episodes_audio folder
repo_root = Path.cwd()
print('notebook cwd =', repo_root)
EPISODES_DIR = None
for p in [repo_root] + list(repo_root.parents):
    candidates = [
        p / 'data' / 'episodes_audio',
        p / 'audio_matcher' / 'data' / 'episodes_audio',
        p / 'episodes_audio',
    ]
    for c in candidates:
        if c.exists():
            EPISODES_DIR = c
            break
    if EPISODES_DIR is not None:
        break

if EPISODES_DIR is None:
    EPISODES_DIR = repo_root / 'audio_matcher' / 'data' / 'episodes_audio'

print('EPISODES_DIR chosen:', EPISODES_DIR)
if not EPISODES_DIR.exists():
    print('EPISODES_DIR does not exist:', EPISODES_DIR)
    files = []
else:
    files = sorted(EPISODES_DIR.rglob('audio.wav'))
    print(f'Found {len(files)} audio.wav files under {EPISODES_DIR}')
    for f in files:
        print('-', f)


notebook cwd = C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\notebooks
EPISODES_DIR chosen: C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio
Found 30 audio.wav files under C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\1TlOcjJodHw\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\326R_Lhua5w\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\3l1lSNQxJA0\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\6uNYmqvF24k\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\audio_matcher\data\episodes_audio\9aLD8SGoPIc\audio.wav
- C:\Users\yihab\Documents\Projects\Antigravity\Video_Analyzer\

In [ ]:
# Build phoneme index from all episodes (multi-file).
import gc
import torch
import numpy as np

pipeline = AudioEmbeddingPipeline()
aligner = PhonemeAligner(device='cpu', whisper_model='base')
files = files

pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)


C:\Users\yihab\miniconda3\envs\video_analyzer\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [ ]:
# final: run phoneme pipeline on chosen song and export
if not files:
    print('No reference files found, skipping pipeline')
else:
    final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
    print(f"output_ms: {len(final_audio)}")
    export_audio(final_audio, 'aligned_output.wav')
    print('Wrote aligned_output.wav')


NameError: name 'pindex' is not defined